In [1]:
# Installer les dépendances indiquées par le projet.
%pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Chargement du corpus
from classes.Corpus import Corpus
from classes.SearchEngine import SearchEngine

corpus = Corpus("Animaux")
corpus.load("corpus_objets.csv")  
print(corpus)

Corpus 'Animaux' : 154 document(s), 149 auteur(s)


In [9]:
import classes.SearchEngine as m
print("Fichier utilisé :", m.__file__)
print(moteur.search("cat", n=1).columns.tolist())

Fichier utilisé : c:\Users\anyak\Projet\classes\SearchEngine.py
['titre', 'auteur', 'source', 'score']


In [8]:
# Instanciation et vérification
moteur = SearchEngine(corpus)
moteur.search("animal behavior", n=3)

Construction de la matrice: 100%|██████████| 154/154 [00:00<00:00, 30786.08it/s]


,titre,auteur,source,score
0,Animal Behavior Analysis Methods Using Deep Le...,Edoardo Fazzari,arxiv,0.2542
1,Models of Animal Behavior as Active Particle S...,Amir Haluts,arxiv,0.1951
2,Collective Animal Behavior from Bayesian Estim...,Alfonso Pérez-Escudero,arxiv,0.1656


In [10]:
# Autres requêtes de contrôle
moteur.search("deep learning", n=3)
moteur.search("xyzabc")   # doit renvoyer un tableau vide

,titre,auteur,source,score


In [11]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML

# Titre en HTML
titre = widgets.HTML(
    value="<h2 style='color:#3e2f23; margin:4px 0;'>Moteur de recherche</h2>"
)
champ_mots = widgets.Text(
    value="",
    placeholder="Entrez vos mots-clés",
    description="Mots clés :",
    layout=widgets.Layout(width="60%"),
)
curseur = widgets.IntSlider(
    value=5, min=1, max=20, step=1,
    description="Nb articles :",
    layout=widgets.Layout(width="60%"),
    style={"description_width": "90px"},
)
filtre_source = widgets.Dropdown(
    options=["Toutes", "reddit", "arxiv"],
    value="Toutes",
    description="Source :",
    layout=widgets.Layout(width="60%"),
)

# La zone de résultats
sortie = widgets.Output()
sortie.add_class("zone-resultats")

# Images décoratives à gauche et à droite (n'importe quel PNG/JPG du projet)
with open("chat.png", "rb") as f:
    octets_image = f.read()

image_gauche = widgets.Image(
    value=octets_image, format="png",
    layout=widgets.Layout(width="110px"),
)
image_droite = widgets.Image(
    value=octets_image, format="png",
    layout=widgets.Layout(width="110px"),
)

# Thème taupe
styles = HTML("""
<style>
    .moteur {
        background-color: #d8cfbf;       /* taupe clair */
        border: 2px solid #b3a68e;
        border-radius: 14px;
        padding: 16px;
        width: 100%;
    }
    .moteur .widget-button {
        border-radius: 10px;
        font-weight: bold;
        background-color: #6d5843 !important;   /* marron */
        color: #f5efe4 !important;
    }
    .moteur .widget-text input { border-radius: 8px; }
    .zone-resultats {
        background-color: #c2b49b;       /* taupe plus sombre */
        border-radius: 10px;
        padding: 10px;
    }
    .zone-resultats, .zone-resultats .dataframe {
        color: #3e2f23;                  /* texte marron foncé */
    }
    .zone-resultats .dataframe,
    .zone-resultats .dataframe th,
    .zone-resultats .dataframe td {
        background-color: #c2b49b !important;
        color: #3e2f23 !important;
        border-color: #a89a80 !important;
    }
</style>
""")
display(styles)

In [ ]:
MOTS_VIDES = {
    "that", "with", "from", "have", "their", "these", "those", "when",
    "where", "this", "what", "which", "will", "would", "there", "been",
    "were", "they", "them", "then", "than", "your", "into", "some",
    "just", "like", "also", "about", "after", "because", "while",
    "more", "other", "such", "only", "over", "much", "here", "well",
    "the", "and", "she", "but", "for", "are", "her", "was", "our",
    "him", "can", "not", "has", "his", "you"
}

def clique_bouton(b):
    with sortie:
        sortie.clear_output()
        if not champ_mots.value.strip():
            print("Entrez au moins un mot-clé.")
            return
        resultats = moteur.search(champ_mots.value, n=curseur.value)
        if filtre_source.value != "Toutes":
            resultats = resultats[resultats["source"] == filtre_source.value]
        if resultats.empty:
            print("Aucun document trouvé.")
        else:
            display(resultats)

def clique_concorde(b):
    with sortie:
        sortie.clear_output()
        if not champ_mots.value.strip():
            print("Entrez un mot pour le concordancier.")
            return
        mot = champ_mots.value.split()[0]
        tableau = corpus.concorde(mot, taille_contexte=30)
        if tableau.empty:
            print(f"Aucune occurrence de « {mot} ».")
        else:
            print(f"{len(tableau)} occurrence(s) de « {mot} » :")
            display(tableau.head(20))

def clique_stats(b):
    with sortie:
        sortie.clear_output()
        freq = corpus.construire_freq()
        top = (
            freq[~freq["mot"].isin(MOTS_VIDES) & (freq["mot"].str.len() > 2)]
            .sort_values("term frequency", ascending=False)
            .head(15)
        )
        plt.figure(figsize=(9, 4), facecolor="#d8cfbf")
        plt.bar(top["mot"], top["term frequency"], color="#6d5843")
        plt.xticks(rotation=45, ha="right", color="#3e2f23")
        plt.title("Les 15 mots les plus fréquents du corpus", color="#3e2f23")
        plt.tight_layout()
        plt.show()

from datetime import datetime

def clique_evolution(b):
    with sortie:
        sortie.clear_output()
        if not champ_mots.value.strip():
            print("Entrez un mot pour suivre son évolution.")
            return
        mot = champ_mots.value.split()[0]

        # Occurrences du mot regroupées par le mois de publication
        series = {}
        for doc in corpus.documents.values():
            nb = corpus.nettoyer_texte(doc.texte).split().count(mot)
            if nb == 0:
                continue
            try:
                date = datetime.fromisoformat(doc.date.replace("Z", "+00:00"))
            except (ValueError, AttributeError):
                continue          # date manquante ou mal formée est ignorée
            periode = date.strftime("%Y-%m")
            series[periode] = series.get(periode, 0) + nb

        if not series:
            print(f"Aucune occurrence de « {mot} ».")
            return

        periodes = sorted(series)
        plt.figure(figsize=(9, 4), facecolor="#d8cfbf")
        plt.plot(periodes, [series[p] for p in periodes],
                 marker="o", color="#6d5843")
        plt.xticks(rotation=45, ha="right", color="#3e2f23")
        plt.title(f"Évolution du mot « {mot} »", color="#3e2f23")
        plt.tight_layout()
        plt.show()

import math

def clique_comparer(b):
    with sortie:
        sortie.clear_output()

        # TF de chaque sous-corpus : chaque source = un gros document
        tf = {"reddit": {}, "arxiv": {}}
        for doc in corpus.documents.values():
            source = doc.getType()
            if source not in tf:
                continue
            for mot in corpus.nettoyer_texte(doc.texte).split():
                tf[source][mot] = tf[source].get(mot, 0) + 1

        # TF-IDF : idf = log(2/df), df = nombre de sous-corpus contenant le mot
        # Un mot présent des deux côtés a un idf nul : il disparaît.
        scores = {"reddit": {}, "arxiv": {}}
        for source in tf:
            for mot, nb in tf[source].items():
                if mot.isalpha() and len(mot) > 3 and mot not in MOTS_VIDES:      # écarte les déchets (http, www...)
                    df = sum(mot in tf[s] for s in tf)  # 1 ou 2
                    scores[source][mot] = nb * math.log(2) if df == 1 else 0

        # Deux graphiques côte à côte
        fig, axes = plt.subplots(1, 2, figsize=(12, 5), facecolor="#d8cfbf")
        for axe, source in zip(axes, ["reddit", "arxiv"]):
            top = sorted(scores[source].items(),
                         key=lambda x: x[1], reverse=True)[:12]
            if top:
                mots_top = [m for m, _ in top][::-1]
                valeurs = [v for _, v in top][::-1]
                axe.barh(mots_top, valeurs, color="#6d5843")
            axe.set_title(f"Mots spécifiques — {source}", color="#3e2f23")
            axe.tick_params(colors="#3e2f23")
            axe.set_facecolor("#c2b49b")
        plt.tight_layout()
        plt.show()

        # Synthèse chiffrée
        communs = set(tf["reddit"]) & set(tf["arxiv"])
        print(f"Mots communs aux deux sources : {len(communs)}")
        print(f"Mots propres à Reddit : {len(set(tf['reddit']) - set(tf['arxiv']))}")
        print(f"Mots propres à arXiv : {len(set(tf['arxiv']) - set(tf['reddit']))}")

In [27]:
bouton = widgets.Button(description="Rechercher")
bouton.on_click(clique_bouton)

bouton_concorde = widgets.Button(description="Concordancier")
bouton_concorde.on_click(clique_concorde)

bouton_stats = widgets.Button(description="Statistiques")
bouton_stats.on_click(clique_stats)
bouton_evolution = widgets.Button(description="Évolution")
bouton_evolution.on_click(clique_evolution)

bouton_comparer = widgets.Button(description="Comparer")
bouton_comparer.on_click(clique_comparer)

ligne_boutons = widgets.HBox(
    [bouton, bouton_concorde, bouton_stats, bouton_evolution, bouton_comparer]
)

# Colonne centrale : titre, champ, curseur, boutons
colonne = widgets.VBox([titre, champ_mots, curseur, filtre_source, ligne_boutons])

# Images à gauche et à droite de cette colonne
ligne_principale = widgets.HBox([image_gauche, colonne, image_droite])

# Zone de résultats sous le tout
interface = widgets.VBox([ligne_principale, sortie])
interface.add_class("moteur")

display(interface)